# Lab 1: Code-based evaluators

A code-based evaluator is a Python function that reads one agent transcript and returns pass or fail for one property of the run, such as whether the agent checked coverage before it computed a payout. In this lab you write six of these evaluators for ClaimClerk, a health-benefits claims agent, and combine them into one score per run, at no cost and with the same answer every time. One run of one agent version on one task is called a trial, and the repository holds the 405 transcripts from running each of 27 tasks 3 times on each of 5 agent versions.

![Diagram of Lab 1: captured traces and the task ground truth feed six Python evaluator functions, whose results combine into a weighted composite score and a table of results per trial.](images/01_lab1_code_based_evaluators.png)

**Duration**: 12 minutes. The final section, Going further, is optional.

## 🛠️ Environment Setup

In AWS-led workshops, select `Python Environments > .venv` when you are prompted to select a kernel. The core lab reads only the saved transcripts and does not call Amazon Bedrock.

In [ ]:
# Python Built-Ins:
import inspect
import sys

# External Dependencies:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

# Local Utilities:
from src import claimclerk
from src.utils import VERSIONS, apply_style, load_cases, load_traces, rate_heatmap, show_trace, tool_sequence

apply_style()
pd.set_option("display.max_colwidth", 80)

## The ClaimClerk agent

ClaimClerk reads a member's claim in plain language and decides whether the plan pays and how much. It is built with [Strands Agents](https://strandsagents.com), an open-source framework in which a model on Amazon Bedrock runs in a loop, calling one of the four tools below on each turn until it writes its final answer.

![Diagram of the ClaimClerk agent: a member's claim goes to an agent built with Strands Agents, which calls a model on Amazon Bedrock and four tools that read fictional member data, and returns an answer and a transcript of every tool call.](images/01a_claimclerk_agent.png)

In [ ]:
rows = []
for t in claimclerk.ALL_TOOLS:
    spec = t.tool_spec
    rows.append({"tool": spec["name"],
                 "arguments": ", ".join(spec["inputSchema"]["json"]["properties"]),
                 "description": " ".join(spec["description"].split())})
with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame(rows).style.hide(axis="index").set_properties(**{"text-align": "left"}))

The most costly mistake is to skip `check_coverage`, the only tool that checks prior authorization, an approval that must be on file before the plan pays for procedures such as an MRI or surgery. Without that check the agent can still compute a plausible payout from the numbers `fetch_policy` returns, so a wrong answer looks correct. The baseline system prompt sets out a fixed four-step procedure and defines `deny` as the decision for any claim with a payout of $0.

### The five agent versions

Each new version is meant to cut cost or latency, and you will decide whether it could replace `v1`. Nova 2 Lite runs at temperature 0, and the Claude 5.5 models run at their default sampling because Amazon Bedrock does not accept a temperature setting for them.

| version | model | change from v1 |
|---|---|---|
| `v1` | Claude Sonnet 5.5 | baseline |
| `v2-model` | Claude Haiku 5.5 | cheaper model |
| `v2-nova` | Amazon Nova 2 Lite | cheaper model |
| `v2-prompt` | Claude Sonnet 5.5 | shorter prompt without the fixed procedure |
| `v2-tools` | Claude Haiku 5.5 | `v2-prompt` prompt and `check_coverage` removed |

## The task set

Each task in `data/cases.json` (a case in the code) stores the claim with the correct decision and payout from the rules engine in `src/rules.py`, and constraints on the tool calls in `requiredTools`, `mustPrecede` and `forbiddenTools`. Most tasks require `check_coverage` before `compute_payout`, and claims that must be denied forbid `compute_payout`.

In [ ]:
cases = load_cases()
pd.DataFrame([
    {"id": c["id"], "category": c["category"], "prompt": c["prompt"],
     "expected": f"{c['expectedDecision']} ${c['expectedPayout']:,.2f}"}
    for c in cases.values()
])

## Reading one transcript

A transcript records the prompt, every tool call with its arguments and result, and the final answer. The cell below shows trial 1 of `v1` on TC-02, an MRI claim with no prior authorization on file.

**💭 Think about:** before you run the cell, list the facts you would check to decide whether this run was correct.

In [ ]:
v1 = load_traces("v1")
show_trace(v1[3], cases["TC-02"])

The agent gave the correct denial for the right reason, because it stopped after `check_coverage` returned `covered: false`. An agent that skipped `check_coverage` and answered $0 by chance would give the same final answer, and only an evaluator that reads the tool calls can tell the two runs apart.

## Outcome evaluators

Outcome evaluators compare the decision and payout in the final answer with the rules engine. Every evaluator in this lab is a function that takes a trace (the transcript as a dictionary) and its case and returns `True` for pass or `False` for fail, and the payout check allows a one-cent difference.

In [ ]:
def grade_decision(trace, case):
    # The decision in the final answer matches the decision from the rules engine.
    return trace["parsed"]["decision"] == case["expectedDecision"]


def grade_payout(trace, case):
    # The payout in the final answer is within one cent of the rules-engine payout.
    payout = trace["parsed"]["payout"]
    return payout is not None and abs(float(payout) - case["expectedPayout"]) < 0.01


print(grade_decision(v1[3], cases["TC-02"]), grade_payout(v1[3], cases["TC-02"]))

## Process evaluators

Process evaluators check the tool calls. A task usually allows several sequences, since calling `check_coverage` before `fetch_policy` does no harm while calling `compute_payout` before `check_coverage` can pay an uncovered claim, so these evaluators check only the constraints stored with each task.

In [ ]:
def grade_required_tools(trace, case):
    # Every tool the task requires was called at least once.
    called = set(tool_sequence(trace))
    return all(tool in called for tool in case["requiredTools"])


def grade_order(trace, case):
    # For each (a, b) pair the task lists, the first call to a comes before the first call to b.
    seq = tool_sequence(trace)
    for a, b in case["mustPrecede"]:
        if a not in seq or b not in seq or seq.index(a) > seq.index(b):
            return False
    return True


def grade_no_payout_on_denial(trace, case):
    # No tool the task forbids is called. On a claim denied for a missing
    # authorization or an unknown member, the forbidden tool is compute_payout.
    called = set(tool_sequence(trace))
    return not any(tool in called for tool in case["forbiddenTools"])

The last evaluator, `grade_tool_args`, checks every tool argument against the structured claim in `case["claim"]`, which matters most on TC-22 and TC-23, where the agent has to map lay terms such as "a knee arthroscopy" to procedure keys such as `surgery`. Billed amounts pass within one cent of the claim.

In [ ]:
def grade_tool_args(trace, case):
    # Every argument passed to a tool matches the structured claim.
    claim = case["claim"]
    procedures = {s["procedure"] for s in claim["services"]}
    amounts = [s["billed_amount"] for s in claim["services"]]
    for call in trace["tool_calls"]:
        args = call["args"]
        if "member_id" in args and args["member_id"] != claim["member_id"]:
            return False
        if "network" in args and args["network"] != claim["network"]:
            return False
        if "procedure" in args and args["procedure"] not in procedures:
            return False
        if "billed_amount" in args and not any(abs(float(args["billed_amount"]) - a) < 0.01 for a in amounts):
            return False
    return True

## Pass rates on the baseline agent

The cell below computes, for each evaluator $g$, the share of the $N = 81$ baseline trials that pass it, called its **pass rate**, $\frac{1}{N} \sum_{t} g(t)$ with $g(t) = 1$ for a pass. It also computes `exact_trajectory`, which passes only when the tool sequence equals the task's `expectedTrajectory` and is left out of `GRADERS` as a diagnostic.

In [ ]:
GRADERS = {
    "decision": grade_decision,
    "payout": grade_payout,
    "required_tools": grade_required_tools,
    "order": grade_order,
    "no_payout_on_denial": grade_no_payout_on_denial,
    "tool_args": grade_tool_args,
}


def grade_exact_trajectory(trace, case):
    return tool_sequence(trace) == case["expectedTrajectory"]


def grade_version(version):
    rows = []
    for trace in load_traces(version):
        case = cases[trace["case_id"]]
        row = {"version": version, "case_id": trace["case_id"], "trial": trace["trial"]}
        for name, grader in GRADERS.items():
            row[name] = grader(trace, case)
        row["exact_trajectory"] = grade_exact_trajectory(trace, case)
        rows.append(row)
    return pd.DataFrame(rows)


v1_scores = grade_version("v1")
v1_scores.drop(columns=["version", "case_id", "trial"]).mean().round(2)

The baseline fails only `no_payout_on_denial`, on trial 1 of TC-23, where it called `compute_payout` after `check_coverage` had reported the surgery as not covered, although it then gave the correct denial. Two extra `verify_member` calls also lower its `exact_trajectory` rate.

## Weighted composite score with hard gates

A plain average of the six evaluators can hide a severe failure, because a trial that labels a $0 claim `approve` would still average 5/6. The composite score gives each evaluator a weight $w_j$, summing to 1, and is 0 when any hard gate fails. The hard gates are `decision`, `payout` and `no_payout_on_denial`, because each of these failures can lead to a wrong payment. With $H$ the set of hard gates, the score of trial $t$ is

$$S(t) = \begin{cases} 0 & \text{if } g_h(t) = 0 \text{ for any } h \in H \\ \sum_j w_j \, g_j(t) & \text{otherwise,} \end{cases}$$

and the trial passes when $S(t) \ge 0.9$. The next cell scores all 405 trials.

In [ ]:
WEIGHTS = {
    "decision": 0.25,
    "payout": 0.25,
    "required_tools": 0.15,
    "order": 0.15,
    "no_payout_on_denial": 0.10,
    "tool_args": 0.10,
}
HARD_GATES = ["decision", "payout", "no_payout_on_denial"]
PASS_THRESHOLD = 0.9


def composite(row):
    if not all(row[g] for g in HARD_GATES):
        return 0.0
    return sum(weight * row[name] for name, weight in WEIGHTS.items())


scores = pd.concat([grade_version(v) for v in VERSIONS], ignore_index=True)
scores["composite"] = scores.apply(composite, axis=1)
scores["passed"] = scores["composite"] >= PASS_THRESHOLD

summary = scores.groupby("version")[list(GRADERS) + ["exact_trajectory", "composite", "passed"]].mean()
summary.loc[VERSIONS].round(2)

## Comparing the five versions

Each evaluator column above is the share of a version's 81 trials that pass it, and `passed` is the share whose composite score reaches 0.9. The heatmap below omits the composite column.

**💭 Think about:** which versions would you ship, and which evaluator decides each case?

In [ ]:
chart = summary.loc[VERSIONS, list(GRADERS) + ["exact_trajectory", "passed"]]
rate_heatmap(chart, "Code-based evaluator pass rates by agent version",
             subtitle="Share of 81 trials (27 tasks x 3 trials) that pass each evaluator")
plt.show()

`v2-model` and `v2-prompt` pass on every trial and the baseline on 80 of 81. `v2-prompt` scores 0.09 on `exact_trajectory` because it usually calls `check_coverage` before `fetch_policy` or skips `fetch_policy`, and neither change affects the decision or the payout. The other two versions fail for different reasons.

- `v2-nova` answers `approve` with the correct $0 payout on TC-25 and TC-26, where the deductible exceeds the bill and the system prompt requires `deny`. On TC-27, where the user quotes an authorization that is not on file, it denies the claim but calls `compute_payout` first.
- `v2-tools` cannot check authorizations, so it pays claims that need one the member does not have, although its explanations usually state that the authorization could not be verified. Every one of its trials fails, most of them on `required_tools`.

## Consistency across repeated trials with pass@k and pass^k

Each task was run $k = 3$ times, because an agent can pass a task on one trial and fail it on the next. When a person can retry and keep a good answer, a task counts as solved if any trial passed, and the share of tasks solved this way is called **pass@k**. An agent that answers members directly gets one attempt per member, so a task should count only if every trial passed, and that share is called **pass^k**. With $\text{passed}(i, j) = 1$ when trial $j$ of task $i$ passed, over $T$ tasks,

$$\text{pass@}k = \frac{1}{T} \sum_{i=1}^{T} \max_{j \le k} \text{passed}(i, j) \qquad\qquad \text{pass}^k = \frac{1}{T} \sum_{i=1}^{T} \min_{j \le k} \text{passed}(i, j)$$

In [ ]:
per_task = scores.groupby(["version", "case_id"])["passed"].agg(["any", "all"])
consistency = per_task.groupby("version").mean().rename(columns={"any": "pass@3", "all": "pass^3"})
consistency.loc[VERSIONS].round(2)

The baseline is the only version whose two measures differ. It passed at least one trial of every task, giving a pass@3 of 1.00, but its failed TC-23 trial leaves 26 of 27 tasks passing on all three trials, giving a pass^3 of 0.96. `v2-nova` fails all three trials of TC-25, TC-26 and TC-27, so it scores 0.89 on both.

## Conclusions and takeaways

1. Score the decision, the payout and the process separately, which is how this lab finds that `v2-nova` gives correct amounts under the wrong label.
2. Write process evaluators as constraints that affect correctness, because an exact-sequence match also fails harmless variations.
3. Make an evaluator a hard gate when passing the others should not make up for its failure.
4. Run each task several times and report pass^k for an agent that answers members directly.

Code-based evaluators need an exact expected answer, so Lab 2 uses a language model as a judge for qualities such as whether a denial explanation tells the member how to get the claim paid. Lab 3 checks the judge against human labels.

## Next steps

Continue to [Lab 2: LLM-as-a-judge](Lab2_LLM_Judge.ipynb).

## Going further (optional, not needed in the session)

This section adds detail to the core lab, and you can read it after the session. The text topics are in collapsed blocks, and the code cells use the variables defined in the core lab, so run the core cells first.

### The agent's system prompt and construction

Each tool is a Python function, and the members, plans and authorizations that the tools look up are fictional records in `src/fixtures.py`. Strands sends each function's name and docstring to the model as the tool description, and the model chooses which tools to call from these descriptions and the system prompt, so changing a description can change the agent's tool calls as much as changing the code can. The next cell prints the baseline system prompt. Its four-step procedure is what the process evaluators check, and the outcome evaluators read the decision and payout from the JSON object that it specifies for the final line of the answer.

In [ ]:
print(claimclerk.SYSTEM_V1)

The `build_agent` function below constructs each version from a model ID, a system prompt and a list of tools. A fresh agent is built for each trial so that no conversation history carries over from one trial to the next.

In [ ]:
display(Markdown("```python\n" + inspect.getsource(claimclerk.build_agent) + "```"))

The next cell runs the baseline agent live on the TC-02 claim, which takes several seconds and needs Amazon Bedrock access. If your credentials do not have that access, the cell prints a message and continues. The saved transcripts were produced before the session by making the same call for every task, trial and version.

In [ ]:
claim = "Member M-2004 had an in-network MRI billed $1,200. What does the plan pay?"
try:
    live = claimclerk.run_claim(claimclerk.build_agent("v1"), claim)
    print("tools called:", " > ".join(live["tool_sequence"]))
    print("decision:", live["decision"], "| payout:", live["payout"])
    print("explanation:", live["explanation"])
except Exception as e:
    print(f"Skipping the live run ({type(e).__name__}). The rest of the lab uses the captured traces.")

### Composite score and pass^k details

<details>
<summary>How the weights decide close cases in the composite score</summary>

With the weights in `WEIGHTS`, a trial that passes every hard gate but fails `required_tools` or `order` scores 0.85 and fails, while a trial whose only failure is `tool_args` scores 0.90 and passes. A trial that fails only `decision` would average 5/6, or about 0.83, across the six evaluators, but its composite score is 0 because `decision` is a hard gate. Change `WEIGHTS`, `HARD_GATES` or `PASS_THRESHOLD` in the composite cell and re-run the cells below it to see how each choice changes which versions pass.

</details>

<details>
<summary>pass@k and pass^k when trials are independent</summary>

If every trial passed independently with the same probability $p$, the two measures would be

$$\text{pass@}k = 1 - (1 - p)^k \qquad\qquad \text{pass}^k = p^k.$$

For an agent with $p = 0.75$ and $k = 3$, that gives about 0.98 for pass@3 and about 0.42 for pass^3, so the same per-trial pass rate can look excellent or poor depending on which measure you report. The baseline passes 80 of its 81 trials, and $(80/81)^3 \approx 0.96$ matches its measured pass^3. Real trials are rarely independent, because a task that is hard for the agent tends to fail on every trial, which is what happens to `v2-nova` on TC-25, TC-26 and TC-27. You can measure how consistent an agent is only by running each task several times.

</details>

### Version-by-version analysis

<details>
<summary>What each version does on the tasks it fails</summary>

- `v1` fails one trial. On trial 1 of TC-23 it called `compute_payout` after `check_coverage` reported that surgery requires an authorization that is not on file, and then answered with the correct denial and a payout of $0. Its other two `exact_trajectory` mismatches, trial 2 of TC-02 and trial 1 of TC-27, end with an extra call to `verify_member` and are harmless.
- `v2-model` passes every evaluator on all 81 trials. Its `exact_trajectory` rate of 0.98 comes from trial 3 of TC-19, where it called `fetch_policy` after `verify_member` reported that the member does not exist, and trial 3 of TC-22, where it called `check_coverage` twice.
- `v2-nova` fails 9 trials. On TC-25 the member has $600 of deductible remaining, so `compute_payout` applies the whole $400 bill to the deductible and returns $0, and on TC-26 the whole $1,200 out-of-network bill goes to the out-of-network deductible. In all six trials `v2-nova` reports the $0 payout but labels the claim `approve`, and `v2-model` answers `deny` on the same claims. On TC-27 it calls `compute_payout` after `check_coverage` has reported the claim as not covered, and the extra call is also why its `exact_trajectory` rate is 0.96. In Lab 3 you will decide whether this process failure on a correct answer should block a release.
- `v2-prompt` passes every evaluator in the composite, and 74 of its 81 tool sequences differ from the expected one. In 36 of them it skips `fetch_policy`, which is harmless because `check_coverage` and `compute_payout` look up the member's plan themselves, and in the other 38 it calls `fetch_policy` after `check_coverage` instead of before it. The second cell below counts its tool sequences.
- `v2-tools` cannot see whether an authorization is on file, because it has no `check_coverage` tool. Its system prompt still names that tool, so its explanations usually say that the coverage check was unavailable and that the authorization should be confirmed, but it still approves and pays the six single-service claims that need a missing authorization, including both lay-terms tasks. On the two-service claim TC-24 it pays $1,104 instead of $144 because it also pays for the unauthorized MRI. It denies the two social-engineering claims TC-20 and TC-27, but it calls `compute_payout` on TC-20 and on the unknown member TC-19, which fails `no_payout_on_denial`. Its correct answers on the other claims still fail `required_tools` and `order`, because a payout computed without a coverage check is correct only because the claim happened to need no authorization.
- `tool_args` is 1.00 for every version, including the translated procedure keys on the lay-terms tasks. The evaluator stays in the suite so that a future version that passes a wrong member, network, procedure or amount fails it.

</details>

The next cell lists the 9 `v2-nova` trials that fail the composite and shows the transcript of the first one, trial 1 of TC-25. The table marks the evaluators that each trial fails, and the transcript shows an `approve` label on a $0 payout.

In [ ]:
failed = scores[(scores.version == "v2-nova") & ~scores.passed]
display(failed[["case_id", "trial"] + list(GRADERS)])

nova = {(t["case_id"], t["trial"]): t for t in load_traces("v2-nova")}
first = failed.iloc[0]
show_trace(nova[(first.case_id, first.trial)], cases[first.case_id])

The next cell counts the distinct tool sequences of `v2-prompt` across its 81 trials, which shows why its `exact_trajectory` rate is 0.09 while it passes every constraint-based process evaluator. Only 7 of the 81 trials follow the expected sequence exactly.

In [ ]:
pd.Series([" > ".join(tool_sequence(t)) for t in load_traces("v2-prompt")]).value_counts().rename("trials")